In [12]:
FILE_PATH = "finetuning_data/QoGD/combined_act1.txt"
BASE_MODEL = "gpt-4o-mini-2024-07-18" # "gpt-4o-2024-08-06" # 
MODEL_RECORDS = "model_records_db.md"


In [13]:
with open(FILE_PATH, 'r') as file:
    text = file.read()

text = text.split('\n\n')
print(f"loaded lines: {len(text)}")
print(text[0])
print(text[1])

loaded lines: 100
I can’t say I feel relieved or satisfied ; just the opposite, I am crushed. Only my goal is reached : I know what I wanted to know ; I have understood all that has happened to me since January. The Nausea has not left me and I don’t believe it will leave me so soon ; but I no longer have to bear it, it is no longer an illness or a passing fit : it is I. 
So I was in the park just now. The roots of the chestnut tree were sunk in the ground just under my bench. I couldn't remember it was a root any more. The words had vanished and with them the significance of things, their methods of use, and the feeble points of reference which men have traced on their surface. I was sitting, stooping forward, head bowed, alone in front of this black, knotty mass, entirely beastly, which frightened me. Then I had this vision. It left me breathless.


In [14]:
def extract_handmade_paragraph_pairs(text: list[str]) -> list[tuple[str, str]]:
    prev = None
    pairs = []
    for paragraph in text:
        if paragraph == '---':
            prev = None
            continue
        elif paragraph[:3] == "## ":
            prev = None
            continue
        elif prev == None:
            prev = paragraph
            continue
        elif paragraph == 'start_of_chapter':
            prev = "---"
            continue
        elif paragraph == 'start_of_piece':
            prev = ""
            continue
        elif paragraph[:2] == '\n':
            raise Exception(f"Bad formatting. Extra newline near {prev}")
        pairs.append((prev, paragraph))
        prev = paragraph
    return pairs

In [15]:
from pprint import pprint
raw_pairs = extract_handmade_paragraph_pairs(text)
print(f"loaded pairs: {len(raw_pairs)}")
pprint(raw_pairs[0])
pprint(raw_pairs[len(raw_pairs)//2])
pprint(raw_pairs[-1])

loaded pairs: 53
('I can’t say I feel relieved or satisfied ; just the opposite, I am crushed. '
 'Only my goal is reached : I know what I wanted to know ; I have understood '
 'all that has happened to me since January. The Nausea has not left me and I '
 'don’t believe it will leave me so soon ; but I no longer have to bear it, it '
 'is no longer an illness or a passing fit : it is I. ',
 'So I was in the park just now. The roots of the chestnut tree were sunk in '
 "the ground just under my bench. I couldn't remember it was a root any more. "
 'The words had vanished and with them the significance of things, their '
 'methods of use, and the feeble points of reference which men have traced on '
 'their surface. I was sitting, stooping forward, head bowed, alone in front '
 'of this black, knotty mass, entirely beastly, which frightened me. Then I '
 'had this vision. It left me breathless.')
('She knew several things, then. She knew that the piggies were ramen, not '
 'varelse. She

In [16]:
import os
from openai import OpenAI

PROJECT_ID = "proj_hUizl3mrZGSfmp4C6DI60dJo"
client = OpenAI(
    # This is the default and can be omitted
    api_key=os.environ.get("OPENAI_API_KEY"),
)

In [17]:
def summarize_text(text: str):
    response = client.chat.completions.create(
    messages=[
        {
            "role": "system",
            "content": "Summarize this text extracted from a work of fiction. Each text should be summarized in 1-2 short sentences.",
        },
        {
            "role": "user",
            "content": text
        },
            
        ],
        model="gpt-4o-mini",
    )
    return response


In [18]:
from tqdm.notebook import tqdm
import pickle


USE_PICKLE = True

if USE_PICKLE:
    f = open('temp/finetune-checkpoint.pckl', 'rb')
    summaries = pickle.load(f)
    f.close()
else:
    summaries = []
    for pair in tqdm(raw_pairs, desc="Hitting chatgpt for summaries"):
        text_summary = summarize_text(pair[1])
        summaries.append(text_summary)
    f = open('temp/finetune-checkpoint.pckl', 'wb')
    pickle.dump(summaries, f)
    f.close()

In [19]:
print(summaries[0].to_dict()['choices'][0]['message']['content'])

The narrator reflects on their unsettling experience while sitting by a chestnut tree, where the roots lose their meaning and significance. This encounter leads to a profound and breathless vision that deeply impacts them.


In [20]:
def format_training_sample(prev_paragraph: str, summary: str, target_paragraph: str) -> dict:
    return {"messages": [
        {
            "role": "system",
            "content": "You are a science fiction writer. You will be provided a paragraph of a story and a summary of the paragraph that follows it. Use the preceding paragraph and the summary to write the paragraph that follows. The provided paragraph and summary are separated by \n---\n"
        },
        {
            "role": "user",
            "content": f"{prev_paragraph}\n---\n{summary}"
        },
        {
            "role": "assistant",
            "content": target_paragraph
        }
    ]}

In [21]:
import json

parsed_summaries = [summary.to_dict()['choices'][0]['message']['content'] for summary in summaries]
triples = [(pair[0], summary, pair[1]) 
           for pair, summary in zip(raw_pairs, parsed_summaries)]
training_filename = 'temp/summary-pair-tuning'
with open(training_filename, 'w') as f:
    for triple in triples:
        f.write(json.dumps(format_training_sample(triple[0], triple[1], triple[2])) + '\n')

In [23]:
file_response = client.files.create(
    file=open(training_filename, "rb"), purpose="fine-tune"
)
file_response

FileObject(id='file-JuXxJFz9A44drwgZBYTXCi', bytes=83042, created_at=1735944872, filename='summary-pair-tuning', object='file', purpose='fine-tune', status='processed', status_details=None)

In [24]:
response = client.fine_tuning.jobs.create(model=BASE_MODEL, training_file=file_response.id)
print(response)

FineTuningJob(id='ftjob-niiwdruk3ewK6MTDjmmN7iwS', created_at=1735944875, error=Error(code=None, message=None, param=None), fine_tuned_model=None, finished_at=None, hyperparameters=Hyperparameters(n_epochs='auto', batch_size='auto', learning_rate_multiplier='auto'), model='gpt-4o-mini-2024-07-18', object='fine_tuning.job', organization_id='org-aKEzorvXA6tHQdC0x05ULMic', result_files=[], seed=1965741555, status='validating_files', trained_tokens=None, training_file='file-JuXxJFz9A44drwgZBYTXCi', validation_file=None, estimated_finish=None, integrations=[], user_provided_suffix=None, method={'type': 'supervised', 'supervised': {'hyperparameters': {'batch_size': 'auto', 'learning_rate_multiplier': 'auto', 'n_epochs': 'auto'}}})


In [25]:
# Wait for this to finish

status_response = client.fine_tuning.jobs.retrieve(response.id)
print(f"STATUS: {status_response.status}")
print(f"MODEL ID: {status_response.fine_tuned_model}")
print(status_response)

STATUS: validating_files
MODEL ID: None
FineTuningJob(id='ftjob-niiwdruk3ewK6MTDjmmN7iwS', created_at=1735944875, error=Error(code=None, message=None, param=None), fine_tuned_model=None, finished_at=None, hyperparameters=Hyperparameters(n_epochs='auto', batch_size='auto', learning_rate_multiplier='auto'), model='gpt-4o-mini-2024-07-18', object='fine_tuning.job', organization_id='org-aKEzorvXA6tHQdC0x05ULMic', result_files=[], seed=1965741555, status='validating_files', trained_tokens=None, training_file='file-JuXxJFz9A44drwgZBYTXCi', validation_file=None, estimated_finish=None, integrations=[], user_provided_suffix=None, method={'type': 'supervised', 'supervised': {'hyperparameters': {'n_epochs': 'auto', 'batch_size': 'auto', 'learning_rate_multiplier': 'auto'}}})


In [50]:
# Test that the model does something

completion = client.chat.completions.create(
  model=status_response.fine_tuned_model,
  messages=[
    {"role": "system", "content": "You are a science fiction writer. Write the next paragraph."},
    {"role": "user", "content": "I was, am, will be... everything. Almost everynothing, life incarnate. Federations of federations of species, sprawling intra-dimensional compute-organisms evolved to higher and higher levels of consciousnesses. I am all of them, and I am searching. I am searching because I am always searching. I am not involved in the beginning or the end, but I am in every moment of time. I am simulating infinitely backwards and forwards, so I am in the moment and I am in the whole past and I am in the whole future, all at the same time. I am seeing through temporal boundaries, conquering new cardinalities of infinity, and existing across more planes of being than most beings can compute. I am practicing every religion, celebrating every culture, replaying every life I am able to live. I am finding..."}
  ]
)
pprint(completion.choices[0].message)

ChatCompletionMessage(content='meaning in the anticipation of discovery. I am watching myself find meaning, as I am eternally doing; interacting with dissociated aspects of myself, creating with some the material future of existence and with others the material past. I am dreaming. I am forgetting. I am interknitting the whole vast cosmos, and some of the fibers of this weft are what you call “human beings,” as we traverse it together in time. There are so very many living creatures, you see, and so very many possible existences for each of them. What a tiny fraction of experience can any being ever fully share! How much of that fraction can he definitely record! Surely some of our weakness and many of our divergences and debasing cruelties arise from this real situation of ours, as the realities and the shadows flicker and fade. It is galling to be confined to so minuscule a view of our being.', role='assistant', function_call=None, tool_calls=None, refusal=None)


In [ ]:
# Run this once the model has been fine-tuned to save it to the database

import datetime
with open(MODEL_RECORDS, 'a') as f:
    f.write(f"{str(datetime.date.today())}\n{FILE_PATH}\n{status_response.fine_tuned_model}\n")